# NNUE trainer on Colab — GPU via PyTorch

**Runtime > Change runtime type > T4 GPU** first. That restarts the VM and wipes
`/content`, so do it before anything else.

There is **no build step**. PyTorch ships precompiled CUDA kernels and is preinstalled on
Colab, so no `nvcc`, no cgo, no linker.

Python only does the optimization. Everything validated stays in the Go binary:

| Step | Tool |
|---|---|
| corpus → `.ntc` | Go `trainer --pack` (split frozen inside the file) |
| `.ntc` → `.ntw` on GPU | **`train_gpu.py`** (this notebook) |
| `.ntw` → `nnue_weights.h` | Go `trainer --load … --convert` |
| parity gate | Go + C, `parity.ps1` |

**Pack the corpus at home first** — do not upload 3 GB of text:

```powershell
.\trainer_win_x64.exe --variant=filipino --pack --out-corpus data\filipino.ntc
```

Verified locally: the `.ntc` reader, the mirror table, Go-compatible rounding, and the
`.ntw` writer all match the Go implementation byte-for-byte, and a Python-written `.ntw`
passes the 350-board 3-way parity gate.

In [ ]:
# 1 - confirm a GPU is attached and torch can see it.
# If this says False, the runtime is CPU-only:
#   Runtime > Change runtime type > T4 GPU  (restarts the VM, wipes /content)
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

from google.colab import drive; drive.mount('/content/drive')

# Only two files are needed: train_gpu.py and the packed corpus.
!mkdir -p /content/nt
!cp /content/drive/MyDrive/nnue/train_gpu.py /content/nt/
!cp /content/drive/MyDrive/nnue/filipino.ntc /content/corpus.ntc
!ls -lh /content/nt/train_gpu.py /content/corpus.ntc

In [ ]:
# 2 - smoke test: 2 epochs, so a mistake costs 30 seconds instead of 40 minutes.
# Watch that valid MSE drops below the anchor-only baseline printed above it.
!python3 /content/nt/train_gpu.py --corpus /content/corpus.ntc \
    --h 256 --epochs 2 --out /content/smoke.ntw

In [ ]:
# 3 - the real run. --log is not optional: the holdout MSE is the number that
# decides whether the net ships, and it has been lost twice already.
!python3 /content/nt/train_gpu.py --corpus /content/corpus.ntc \
    --h 256 --epochs 40 --batch 8192 --lr 1e-3 --opt adam --seed 42 \
    --out /content/nn_h256.ntw --log /content/train.log

In [ ]:
# 4 - bring the weights home. Do NOT install the header generated anywhere but
# your own machine: on Windows run
#     trainer_win_x64.exe --variant=filipino --load nn_h256.ntw --convert --outh nnue_weights.h
#     .\parity.ps1 -Bin nn_h256.ntw -Variant filipino      # MUST print PARITY: PASS
# then probe_accbound, then depth-10 games. MSE is necessary, not sufficient.
from google.colab import files
for f in ('nn_h256.ntw', 'train.log'):
    files.download('/content/' + f)

In [ ]:
# 5 - bring it home. Re-validate on the Windows box before installing:
#     tools\parity.ps1  and  probe_accbound.exe
# The artifact is always re-checked on the machine that ships it.
from google.colab import files
for f in ('nnue_weights.h', 'nn_h256.ntw', 'train.log'):
    files.download('/content/' + f)